# 238 — over-factor and filter, and reliability-based electrode selection

Fit **more** components than any criterion supports, then keep only the ones that
survive a stability test — the strategy Norman-Haignere et al. (2019) used, with the
filter changed to target the failure this dataset actually has.

**They** fitted 20 components and restricted their analyses to 14: stable across 1000
random initialisations (median r > 0.9 against the next 99 best, Fig 1G) and not
driven by one subject (max normalised subject weight < 0.5, Fig 1F).

**The critical detail: the MODEL stayed at 20.** They did not refit at 14 and never
claimed 14 components explain the data. In NMF the components are fitted jointly, so
deleting some and keeping the rest leaves something that reconstructs nothing. The
filter restricts *interpretation*, not the model. This notebook keeps that discipline.

**The change:** their initialisation filter catches components that move between
random starts. The primary filter here is the **bootstrap** — resample the electrodes,
refit, ask whether the component comes back — because that is the failure this data
has. Initialisation and patient filters are reported alongside.

## Results so far, on cnmf / concat_hg

| K | interpreted | bootstrap pass | init pass | patient pass | best bootstrap J |
|---|---|---|---|---|---|
| 4 | **0/4** | 0 | 4 | 4 | 0.59 |
| 6 | **0/6** | 0 | 6 | 6 | 0.58 |
| 12 | **0/12** | 0 | 6 | 11 | 0.55 |
| 16 | **0/16** | 0 | 5 | 13 | 0.55 |
| 20 | **0/20** | 0 | 4 | 16 | 0.52 |

**Nothing clears Hennig's 0.60 at any K**, and it degrades as K rises (median J
0.53 → 0.38). Two things worth noting: the initialisation filter *does* start to bite
at high K (6/12, 5/16, 4/20 — the optimiser becomes unstable), and so does the patient
filter. Neither bites at K=7, which is why they looked toothless at first.

## Now runs on any method and either electrode set

Set `METHOD` and `FEATURE_SET` below. The open question is whether the hard
partitions on the **ungated** set behave differently: FIG C.9 shows k-means and Ward
both build a cluster that is 74–78% electrodes the gate would have removed, and that
cluster may well be the most bootstrap-stable thing in the data — which would be an
uncomfortable result worth having.

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
sys.path.insert(0, "functions")
import lf_decompose as LD
import measure_cluster_stability as MS     # resolve(), fit_any(), patient_share()

# ── what to run ───────────────────────────────────────────────────────────────
METHOD      = "cnmf"           # "cnmf" | "kmeans" | "hierarchical"
FEATURE_SET = "concat_hg"      # "concat_hg" (gated) | "concat_hg_all" (ungated)
K_OVER      = 10               # ask for MORE than any criterion supports

N_BOOT   = 25      # bootstrap resamples
N_SEEDS  = 10      # random restarts
BOOT_MIN = 0.60    # Hennig 2007: < 0.60 is not a real cluster
INIT_MIN = 0.75
PAT_MAX  = 0.50    # Norman-Haignere Fig 1F

OUT = Path("outputs/clustering/comparison"); OUT.mkdir(parents=True, exist_ok=True)
TAG = f"{METHOD}_{FEATURE_SET}_K{K_OVER}"

RUN, SPACE = MS.resolve(METHOD, FEATURE_SET)
print(f"{METHOD} / {FEATURE_SET}  ->  {RUN.name}")
print(f"fits in {SPACE} space, over-factoring to K = {K_OVER}")

## 1. Fit the over-factored model

**Each method is fitted in the space it actually uses.** `X_train.npy` is raw dB;
convex NMF unit-norms each electrode before fitting, k-means and Ward do not. Scoring
all three in dB is the error that made the first version of FIG C.7 wrong, and every
measure below — silhouette included — is space-dependent.

In [ ]:
X  = np.load(RUN / "X_train.npy").astype(float)
A  = MS.unit(X) if SPACE == "unit-norm" else X
lab_df = pd.read_csv(RUN / "labels.csv")
patients = lab_df["patient_id"].astype(str).to_numpy()
n = len(A)

# the gate flag rides along in labels.csv; on a gated run every row is True
gate = None
if "n_high_activity" in lab_df.columns:
    g = pd.to_numeric(lab_df["n_high_activity"], errors="coerce").fillna(0).to_numpy() > 0
    if (~g).any():
        gate = g

Gn = None
if METHOD == "cnmf":
    G = LD.convex_nmf(A, K_OVER, n_iter=300, random_state=0)[1]
    Gn = G / np.maximum(G.sum(1, keepdims=True), 1e-12)
    base = Gn.argmax(1)
else:
    base = MS.fit_any(A, K_OVER, 0, METHOD)

ids = sorted(set(int(v) for v in base))
members = {j: np.where(base == j)[0] for j in ids}
print(f"{n} electrodes x {X.shape[1]} features")
print("cluster sizes:", [len(members[j]) for j in ids])
if gate is not None:
    print(f"ungated set: {100*(~gate).mean():.1f}% of electrodes were added by lifting the gate")

## 2. Filter A — bootstrap Jaccard (the one that bites)

Resample the electrodes **with replacement**, refit at the same K, and ask how much of
each original cluster is recovered by its best match. Hennig (2007): **> 0.75 stable,
0.60–0.75 a pattern, < 0.60 not a real cluster.**

This is the primary filter. What it does *not* say is that a surviving component is
biologically meaningful — only that it is not an accident of which electrodes were
recorded.

In [ ]:
boot = {j: [] for j in ids}
for b in range(N_BOOT):
    take = np.random.default_rng(1000 + b).integers(0, n, n)
    lb = MS.fit_any(A[take], K_OVER, 0, METHOD)
    votes = {}
    for pos, orig in enumerate(take):
        votes.setdefault(orig, []).append(lb[pos])
    back = np.full(n, -1)
    for orig, v in votes.items():
        back[orig] = np.bincount(v).argmax()
    seen = sorted(set(int(v) for v in back if v >= 0))
    for j in ids:
        boot[j].append(MS.best_jaccard(members[j], back, seen))
    print(f"  bootstrap {b+1}/{N_BOOT}", end="\r")

boot_J = {j: float(np.mean(v)) for j, v in boot.items()}
print(" " * 30)
for j in ids:
    print(f"  c{j:<2} bootstrap J {boot_J[j]:.2f} +/- {np.std(boot[j]):.2f}")

## 3. Filter B — initialisation stability

Refit from different random starts.

**Ward has no random initialisation** — it is deterministic, so this score is 1.0 by
construction and is not evidence of anything. The cell says so rather than letting it
look like a passed test.

In [ ]:
DETERMINISTIC = METHOD == "hierarchical"
if DETERMINISTIC:
    init_J = {j: float("nan") for j in ids}
    print("Ward is deterministic - no random initialisation, so this filter does not apply.")
else:
    init = {j: [] for j in ids}
    for s in range(1, N_SEEDS + 1):
        ls = MS.fit_any(A, K_OVER, s, METHOD)
        for j in ids:
            init[j].append(MS.best_jaccard(members[j], ls, sorted(set(int(v) for v in ls))))
    init_J = {j: float(np.mean(v)) for j, v in init.items()}
    for j in ids:
        print(f"  c{j:<2} initialisation J {init_J[j]:.2f}")

## 4. Filter C — patient specificity

Norman-Haignere Fig 1F: drop a component if one subject holds more than half its
weight. Five of their twenty died this way.

**The quantity differs by method.** Convex NMF has loadings, so this is the share of
component *weight*, matching the paper. A hard partition has no weights, so it is the
share of *members* instead — a related but not identical quantity, and the 0.5
threshold carries over only loosely.

It also barely bites at this cohort size: they had **13 subjects**, this has **27**, so
a >0.5 single-patient share is roughly twice as hard to reach.

In [ ]:
shares = MS.patient_share(base, patients, Gn=Gn)
for j in ids:
    mx, who = shares[j]
    flag = "  <-- one patient dominates" if mx > PAT_MAX else ""
    print(f"  c{j:<2} largest patient share {mx:.3f} ({who}){flag}")

## 5. What survives — and what that licenses

**The model stays at K = K_OVER.** Nothing is deleted, nothing is refitted. The table
says which components are worth *naming*; the reconstruction and every loading still
come from the full model.

If you later write "we found N components", that is wrong. The honest sentence is:
*a K=… decomposition was fitted; N of its components met the stability criteria and
only those are interpreted.*

In [ ]:
rows = []
for j in ids:
    ok_b = boot_J[j] >= BOOT_MIN
    ok_i = True if DETERMINISTIC else (init_J[j] >= INIT_MIN)
    ok_p = shares[j][0] <= PAT_MAX
    r = dict(component=j, n=len(members[j]),
             bootstrap_J=round(boot_J[j], 3), pass_bootstrap=ok_b,
             init_J=None if DETERMINISTIC else round(init_J[j], 3), pass_init=ok_i,
             patient_max=round(shares[j][0], 3), pass_patient=ok_p,
             interpret=bool(ok_b and ok_i and ok_p))
    if gate is not None:
        m = base == j
        r["pct_added"] = round(float(100 * (~gate[m]).mean()), 1)
    rows.append(r)
tab = pd.DataFrame(rows).set_index("component")
display(tab)

keep = tab.index[tab["interpret"]].tolist()
print(f"\n{METHOD} / {FEATURE_SET}  ({SPACE} space)")
print(f"MODEL: K={K_OVER}, unchanged.")
print(f"INTERPRET: {len(keep)} of {K_OVER} -> {keep}")
print(f"  failed bootstrap : {tab.index[~tab['pass_bootstrap']].tolist()}")
if not DETERMINISTIC:
    print(f"  failed init      : {tab.index[~tab['pass_init']].tolist()}")
print(f"  failed patient   : {tab.index[~tab['pass_patient']].tolist()}")
if gate is not None:
    b = tab.sort_values("bootstrap_J", ascending=False)
    print(f"\n  most bootstrap-stable component is c{b.index[0]} "
          f"(J={b.iloc[0]['bootstrap_J']:.2f}, {b.iloc[0]['pct_added']:.0f}% added)")
    print(f"  correlation across components between bootstrap J and % added: "
          f"{np.corrcoef(tab['bootstrap_J'], tab['pct_added'])[0,1]:+.2f}")
tab.to_csv(OUT / f"overfactor_filter_{TAG}.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 3.4), dpi=140)
x = np.arange(len(ids))
ax.bar(x - 0.2, [boot_J[j] for j in ids], width=0.4, label="bootstrap J", color="#4a6fa5")
if not DETERMINISTIC:
    ax.bar(x + 0.2, [init_J[j] for j in ids], width=0.4, label="initialisation J",
           color="#b0b7be")
ax.axhline(BOOT_MIN, color="#c1121f", ls="--", lw=1)
ax.text(len(ids) - 0.4, BOOT_MIN + 0.01, f"Hennig {BOOT_MIN:g}", color="#c1121f",
        fontsize=8, ha="right")
if gate is not None:
    for i, j in enumerate(ids):
        ax.text(i - 0.2, boot_J[j] + 0.02, f"{tab.loc[j,'pct_added']:.0f}%",
                ha="center", fontsize=6.5, color="#5f6a72")
for i, j in enumerate(ids):
    if tab.loc[j, "interpret"]:
        ax.plot(i, 1.02, marker="v", color="#1b7837", ms=7)
ax.set_xticks(x); ax.set_xticklabels([f"c{j}" for j in ids])
ax.set_ylim(0, 1.1); ax.set_ylabel("Jaccard")
ax.set_title(f"{METHOD} / {FEATURE_SET}, K={K_OVER} over-factored then filtered"
             + ("   (% = share added by ungating)" if gate is not None else ""),
             fontsize=10, loc="left")
ax.legend(fontsize=8, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(OUT / f"overfactor_filter_{TAG}.png", dpi=140, bbox_inches="tight",
            facecolor="white")
plt.show()

## 6. Reliability-based electrode selection — BLOCKED, and exactly why

The responsiveness gate thresholds on **amplitude**. That cannot separate a
small-but-repeatable response from a large-but-random one, which is why lifting it
added 1679 electrodes whose mean |HG| was **0.392 dB** — small, but not zero — and why
FIG C.10 finds those electrodes carry the *same* feature signature as the gated ones
(r = 0.75–0.95), only weaker.

The principled replacement is **split-half reliability**: compute each electrode's ERSP
twice from disjoint halves of its trials and correlate. Norman-Haignere do exactly
this — their 271 electrodes are the survivors of **split-half r > 0.2** — and they
reuse the same quantity to noise-correct their variance explained, so it buys two
things at once.

**Why it cannot run today.** The saved cubes are already trial-averaged:

    01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY/<pid>/LM/ERSP_matrix/<cond>/*.npy
    shape (129, 300)   <- frequency x time, no trial axis

**Stage 01 (140/150) has to write them.** The minimal change is two extra cubes per
electrode-condition, from odd and even trials:

    <pid>_<cond>_WM_ERSP_<el>_TN_half1.npy
    <pid>_<cond>_WM_ERSP_<el>_TN_half2.npy

Odd/even rather than first-half/second-half on purpose: it balances drift, fatigue and
block structure across the halves. Norman-Haignere split on odd and even *runs* for the
same reason.

The cell below is written and checks for the halves rather than failing.

In [ ]:
ERSP = Path("../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY")
halves = sorted(ERSP.glob("*/LM/ERSP_matrix/*/*_half1.npy"))
print(f"split-half cubes found: {len(halves)}")

if not halves:
    print("\nBLOCKED - stage 01 has not written split halves yet.")
    print("Re-run 140/150 with the half1/half2 cubes enabled, then this cell works.")
    print("This is a check, not a failure; nothing below depends on it.")
else:
    # Spearman-Brown corrects a half-length correlation up to full test length.
    rel = {}
    for p1 in halves:
        p2 = Path(str(p1).replace("_half1.npy", "_half2.npy"))
        if not p2.exists():
            continue
        a_, b_ = np.load(p1).ravel(), np.load(p2).ravel()
        r = float(np.corrcoef(a_, b_)[0, 1])
        rel[p1.name.replace("_half1.npy", "")] = 2 * r / (1 + r) if r > -1 else np.nan
    rs = pd.Series(rel).dropna()
    print(f"\n{len(rs)} electrode-conditions with a reliability estimate")
    print(rs.describe().round(3).to_string())
    print(f"\nwould survive Norman-Haignere's r > 0.2: {(rs > 0.2).sum()} "
          f"({100*(rs > 0.2).mean():.0f}%)")
    fig, ax = plt.subplots(figsize=(6, 3), dpi=140)
    ax.hist(rs, bins=60, color="#4a6fa5")
    ax.axvline(0.2, color="#c1121f", ls="--", lw=1)
    ax.text(0.2, ax.get_ylim()[1]*0.95, " r > 0.2 (Norman-Haignere)", fontsize=7,
            color="#c1121f", va="top")
    ax.set_xlabel("split-half reliability (Spearman-Brown corrected)")
    ax.set_ylabel("# electrode-conditions")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()
    rs.to_csv(OUT / "electrode_reliability.csv")

## 7. What to run, and what not to conclude

**Suggested order.** `cnmf/concat_hg` is done (0 survivors at K = 4, 6, 12, 16, 20).
Next: `kmeans/concat_hg_all` and `hierarchical/concat_hg_all`, then the gated
equivalents as a baseline.

**A prediction, and an early signal against it.** FIG C.9 shows k-means and Ward both
build a cluster that is 74-78% electrodes the gate would have removed, so the obvious
hypothesis was that this cluster would be the most bootstrap-stable thing in the data -
i.e. that the most reproducible structure in the ungated set is an artefact of
including non-responsive electrodes.

A 3-repetition smoke test on `kmeans/concat_hg_all` at K=10 points the **other** way:
the added-heavy clusters scored LOW (c5, 76% added, J = 0.27; c0, 73% added, J = 0.41)
while the highest was a small cluster with **0% added** (n=50, J = 0.61) - which then
failed the patient filter at 0.84, one patient holding most of it.

Three repetitions is not a result, and the full run at N_BOOT = 25 is what settles it.
But do not run this expecting to confirm the hypothesis: on current evidence the
gate-driven clusters are large and unstable, and what stability exists sits in small
single-patient clusters that the patient filter then removes. The last cell prints the
correlation between bootstrap J and % added, which is the compact version of the test.

**Two things this notebook deliberately does not do.**

It does not refit at the number of survivors - that is a different model, and the
survivors are not a decomposition of anything on their own.

It does not treat surviving as evidence a component is real. Every filter is a
*necessary* condition. Passing all three clears the bar for being worth discussing;
whether it corresponds to anything in the brain is what FIG C.6's feature drivers and
the anatomy are for.